<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/team-head-to-head.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Team Racing Head-to-Head

Team racing regattas sail multiple 3-boat-vs-3-boat matches per round, and the
same two schools often meet more than once across a season (round robins,
conference championships, invitationals). This notebook finds two schools
that have raced each other directly the most, and builds:

- a **match log** — every individual race the two schools sailed against each
  other, with boat-position combos and the winner
- **summary stats** — overall W-L-T record, a per-regatta breakdown, and a
  look at how often the win came from a "strong" position combo
- a **chart** of the head-to-head record over time

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"


## Scrape a season's team racing

Team racing lives in the **spring** season (fall is almost entirely fleet
racing), so we load `s25` (Spring 2025) and narrow to team-scored regattas
with `.team()`. `sailors=False` skips the per-sailor pages we don't need
here. The first run scrapes the season (cached to disk); re-runs are
instant.

In [ ]:
import scraper

SEASON = "s25"

data = scraper.load(SEASON, sailors=False).team()
print(len(data.regattas), "team-racing regattas in", SEASON)

## Find the two schools that met the most

Walk every regatta's teams → rounds → matches and count matches per
**unordered** school pair (each match is recorded from both schools'
perspectives in the data, so a pair's raw count is naturally even — that's
expected, not a bug). No hardcoded slugs: whichever pair met most often wins.

In [ ]:
from collections import Counter

pair_counts = Counter()
for reg in data.regattas:
    for team in reg.teams:
        for rnd in team.rounds:
            for m in rnd.matches:
                if not m.sailed:
                    continue
                pair = tuple(sorted((team.school_slug, m.opponent)))
                pair_counts[pair] += 1

(school_a, school_b), meeting_count = pair_counts.most_common(1)[0]
print(f"Most-met pair: {school_a} vs {school_b} ({meeting_count} recorded race-sides)")

## Build the match log

Each match is duplicated in the data (once from each school's side), so we
dedupe by `(season, regatta slug, race_num)` and present every race
consistently from `school_a`'s perspective (alphabetically first of the two
slugs).

In [ ]:
import pandas as pd

rows = []
seen = set()
for reg in data.regattas:
    for team in reg.teams:
        if team.school_slug not in (school_a, school_b):
            continue
        for rnd in team.rounds:
            for m in rnd.matches:
                if not m.sailed or m.opponent not in (school_a, school_b):
                    continue
                key = (reg.season, reg.slug, m.race_num)
                if key in seen:
                    continue
                seen.add(key)

                # Present from school_a's perspective regardless of which
                # team object we happened to be iterating.
                if team.school_slug == school_a:
                    our_positions, their_positions = m.our_positions, m.their_positions
                    won = m.won
                else:
                    our_positions, their_positions = m.their_positions, m.our_positions
                    won = not m.won and not m.tied

                if m.tied:
                    winner = "tie"
                elif won:
                    winner = school_a
                else:
                    winner = school_b

                rows.append(
                    {
                        "season": reg.season,
                        "regatta": reg.name,
                        "regatta_slug": reg.slug,
                        "round": rnd.name,
                        "race_num": m.race_num,
                        f"{school_a}_positions": our_positions,
                        f"{school_b}_positions": their_positions,
                        "winner": winner,
                    }
                )

matches = (
    pd.DataFrame(rows).sort_values(["season", "regatta_slug", "race_num"]).reset_index(drop=True)
)
print(len(matches), "direct matches between", school_a, "and", school_b)
matches.head(10)

## Summary stats

Overall record, a per-regatta breakdown, and a note on position combos: in
3-on-3 team racing the six finishing positions (1st-6th) split between the
two boats-per-side, so a combo summing to 6 or less (1-2, 1-3, 2-3) is a
"strong" win — the other side is guaranteed worse boats in every remaining
slot.

In [ ]:
record = matches["winner"].value_counts()
wins_a = int(record.get(school_a, 0))
wins_b = int(record.get(school_b, 0))
ties = int(record.get("tie", 0))
print(f"Overall record ({school_a} perspective): {wins_a}-{wins_b}-{ties} (W-L-T)")

by_regatta = (
    matches.groupby(["season", "regatta"])["winner"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
by_regatta

In [ ]:
a_wins = matches[matches["winner"] == school_a]
if len(a_wins):
    combo_sum = a_wins[f"{school_a}_positions"].apply(sum)
    strong = (combo_sum <= 6).mean()
    print(
        f"{school_a} strong-combo (sum <= 6) share of wins: "
        f"{strong:.0%} ({int((combo_sum <= 6).sum())} of {len(a_wins)})"
    )
else:
    print(f"{school_a} has no wins in this head-to-head to analyze combos for.")

## Chart: cumulative record over the season

Cumulative wins for each school across the sequence of matches (in season /
regatta / race order).

In [ ]:
import matplotlib.pyplot as plt

cum_a = (matches["winner"] == school_a).cumsum()
cum_b = (matches["winner"] == school_b).cumsum()

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(matches) + 1), cum_a, marker="o", label=school_a)
plt.plot(range(1, len(matches) + 1), cum_b, marker="o", label=school_b)
plt.xlabel("Match number (chronological order sailed)")
plt.ylabel("Cumulative wins")
plt.title(f"{school_a} vs {school_b} — head-to-head, {SEASON}")
plt.legend()
plt.tight_layout()
plt.show()

---
**Going deeper:** each `TeamRaceTeam.rounds` entry is a `TeamRaceRound` with
its own `wins`/`losses`/`ties` and ordered `matches`, and each
`TeamRaceMatch` carries a `flight` number — enough structure to reconstruct
the full round-robin (who sailed whom, in what order, within each round) or
to build a boat-combo win-rate model across every school, not just one pair.
See `scraper/models.py` for the full shape.